# LTE KPI Degradation Analyzer - Enhanced Version v2.0

## Developed by: Musketeers_Team (ITI Graduation Project 2026)

---

### Addresses Reviewer Feedback:
- Configurable baseline window (7-day, 4-week rolling, custom)
- Minimum baseline value filter per KPI
- Severity weighting for cause ranking
- Statistical significance test (t-test)
- Max/percentile aggregation for failure counters
- Vectorized cause detection for performance
- Multi-sheet Excel support
- Per-cell trend chart in Word report

---

## 1. Import Required Libraries

In [ ]:
import os
import traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from scipy import stats

# Matplotlib for charts
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for notebooks
from matplotlib.figure import Figure

# Word document generation
try:
    from docx import Document
    from docx.shared import Inches, Pt, Cm
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    DOCX_AVAILABLE = True
except ImportError:
    DOCX_AVAILABLE = False
    print("Warning: python-docx not installed. Word report generation will be disabled.")

print("Libraries imported successfully!")

## 2. Global Column Names and Constants

In [ ]:
DATE_COL = "Date"
SITE_COL = "eNodeB Name"
CELL_COL = "Cell Name"
LOCAL_CELL_COL = "LocalCell Id"

CELL_ID_COLS = [
    SITE_COL,
    CELL_COL,
    LOCAL_CELL_COL,
]

# Baseline window modes
BASELINE_MODE_LAST_WEEK = "last_week"
BASELINE_MODE_4WEEK_AVG = "4week_rolling_avg"
BASELINE_MODE_CUSTOM = "custom_range"

print("Constants defined!")

## 3. KPI Configuration

### NEW FIELDS ADDED:
- `min_baseline_value`: filter low-traffic/low-load cells
- `severity_weights`: priority for cause ranking

In [ ]:
KPI_CONFIGS = {
    "DL Traffic": {
        "target_kpi": "(HU) DL Traffic Volume (GBytes)",
        "bad_direction": "low",
        "default_threshold": 30.0,
        "category": "Traffic",
        "output_prefix": "dl_traffic",
        "min_baseline_value": 1.0,  # NEW: Filter cells with < 1 GB baseline traffic
        "related_rules": [
            {"feature": "(HU) Cell DL Average Throughput (Mbps)", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "DL Throughput Degradation", "reason": "Cell DL throughput decreased.", "recommended_action": "Check DL scheduler, bandwidth, CA activation, load balancing, and congestion."},
            {"feature": "(HU) User DL Average Throughput (Mbps)", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "User Throughput Degradation", "reason": "User DL throughput decreased.", "recommended_action": "Check radio quality, PRB load, scheduler behavior, and user distribution."},
            {"feature": "(HU) DL PRB Utilization(%)", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "Capacity / Congestion", "reason": "DL PRB utilization increased.", "recommended_action": "Check load balancing, CA usage, bandwidth, traffic distribution, and capacity expansion."},
            {"feature": "DL Average CQI", "bad_direction": "low", "threshold": 15, "severity": 3, "category": "Radio Quality Issue", "reason": "DL CQI decreased.", "recommended_action": "Check interference, PCI conflict/confusion, antenna tilt, azimuth, and coverage."},
            {"feature": "(HU) DL IBLER(%)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "DL Radio Quality Issue", "reason": "DL IBLER increased.", "recommended_action": "Check interference, CQI, MCS, antenna tilt, and DL power."},
            {"feature": "DL RBLER", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "DL Radio Failure", "reason": "DL RBLER increased.", "recommended_action": "Check DL interference, poor coverage, CQI, MCS, and radio conditions."},
            {"feature": "(HU) PDSCH MCS", "bad_direction": "low", "threshold": 15, "severity": 2, "category": "Poor Modulation Efficiency", "reason": "PDSCH MCS decreased.", "recommended_action": "Check CQI, SINR, interference, antenna direction, and coverage."},
            {"feature": "Availability", "bad_direction": "low", "threshold": 1, "severity": 5, "category": "Availability Issue", "reason": "Cell availability decreased.", "recommended_action": "Check alarms, S1 issue, manual outage, system outage, and site availability."},
            {"feature": "(HU) Cell Unavail Time (s)", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "Cell Unavailability", "reason": "Cell unavailable time increased.", "recommended_action": "Check site outage, power issue, transmission issue, S1 failure, and alarms."},
            {"feature": "L.Traffic.ActiveUser.Dl.Avg", "bad_direction": "low", "threshold": 20, "severity": 1, "category": "Traffic Demand Drop", "reason": "DL active users decreased.", "recommended_action": "Validate if traffic drop is normal demand behavior before RF optimization."},
            {"feature": "MAC CA Traffic Ratio", "bad_direction": "low", "threshold": 20, "severity": 2, "category": "Carrier Aggregation Issue", "reason": "CA traffic ratio decreased.", "recommended_action": "Check CA activation, SCell availability, CA bands, and CA parameters."},
            {"feature": "DL Traffic QCI-9", "bad_direction": "low", "threshold": 20, "severity": 2, "category": "Default Bearer Traffic Drop", "reason": "QCI-9 DL traffic decreased.", "recommended_action": "Check packet data service, APN/data service, user demand, and internet traffic trend."},
            {"feature": "DL_CCE_AllocFail (%)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "Control Channel Congestion", "reason": "DL CCE allocation failure increased.", "recommended_action": "Check PDCCH/CCE utilization, control channel capacity, and scheduler configuration."},
        ],
    },

    "UL Traffic": {
        "target_kpi": "(HU) UL Traffic Volume (GBytes)",
        "bad_direction": "low",
        "default_threshold": 30.0,
        "category": "Traffic",
        "output_prefix": "ul_traffic",
        "min_baseline_value": 0.1,  # NEW
        "related_rules": [
            {"feature": "(HU) Cell UL Average Throughput (Mbps)", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "UL Throughput Degradation", "reason": "Cell UL throughput decreased.", "recommended_action": "Check UL scheduler, UL PRB utilization, uplink interference, and power control."},
            {"feature": "(HU) User UL Average Throughput (Mbps)", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "UL User Throughput Degradation", "reason": "User UL throughput decreased.", "recommended_action": "Check UL radio quality, UL interference, PUSCH MCS, and UL PRB load."},
            {"feature": "(HU)UL PRB Utilization(%)", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "UL Capacity / Congestion", "reason": "UL PRB utilization increased.", "recommended_action": "Check UL capacity, UL scheduling, uplink load, and traffic distribution."},
            {"feature": "(HU) Avg UL Interference(dBm)", "bad_direction": "high", "threshold": 10, "severity": 4, "category": "UL Interference Issue", "reason": "Average UL interference increased.", "recommended_action": "Check external interference, PIM, neighboring cells, and uplink noise rise."},
            {"feature": "L.UpPTS.Interference.Avg(dBm)", "bad_direction": "high", "threshold": 10, "severity": 3, "category": "UL Interference Issue", "reason": "UpPTS interference increased.", "recommended_action": "Check uplink interference source and TDD interference conditions."},
            {"feature": "(HU) UL IBLER(%)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "UL Radio Quality Issue", "reason": "UL IBLER increased.", "recommended_action": "Check UL interference, PUSCH MCS, UE power, and coverage."},
            {"feature": "UL RBLER", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "UL Radio Failure", "reason": "UL RBLER increased.", "recommended_action": "Check UL interference, coverage, UE power, and uplink radio conditions."},
            {"feature": "(HU) PUSCH MCS", "bad_direction": "low", "threshold": 15, "severity": 2, "category": "UL Modulation Efficiency Issue", "reason": "PUSCH MCS decreased.", "recommended_action": "Check UL SINR, interference, UE power control, and uplink coverage."},
            {"feature": "L.Traffic.ActiveUser.UL.Avg", "bad_direction": "low", "threshold": 20, "severity": 1, "category": "UL Traffic Demand Drop", "reason": "UL active users decreased.", "recommended_action": "Validate traffic trend before applying RF action."},
        ],
    },

    "DL Throughput": {
        "target_kpi": "(HU) User DL Average Throughput (Mbps)",
        "bad_direction": "low",
        "default_threshold": 20.0,
        "category": "Integrity",
        "output_prefix": "dl_throughput",
        "min_baseline_value": 5.0,  # NEW: Mbps
        "related_rules": [
            {"feature": "(HU) DL PRB Utilization(%)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "DL Congestion", "reason": "DL PRB utilization increased while DL throughput decreased.", "recommended_action": "Check congestion, load balancing, CA, bandwidth, scheduler, and capacity expansion."},
            {"feature": "DL Average CQI", "bad_direction": "low", "threshold": 15, "severity": 4, "category": "Poor Radio Quality", "reason": "CQI decreased while DL throughput degraded.", "recommended_action": "Check interference, PCI, antenna tilt, azimuth, and coverage."},
            {"feature": "(HU) DL IBLER(%)", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "High DL Retransmission", "reason": "DL IBLER increased while DL throughput degraded.", "recommended_action": "Check BLER, CQI, MCS, DL power, and interference."},
            {"feature": "DL RBLER", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "DL Radio Block Errors", "reason": "DL RBLER increased while DL throughput degraded.", "recommended_action": "Check interference, radio quality, and coverage."},
            {"feature": "(HU) PDSCH MCS", "bad_direction": "low", "threshold": 15, "severity": 3, "category": "Low DL Modulation", "reason": "PDSCH MCS decreased while DL throughput degraded.", "recommended_action": "Check CQI, SINR, interference, and coverage."},
            {"feature": "MAC CA Traffic Ratio", "bad_direction": "low", "threshold": 20, "severity": 2, "category": "Low CA Usage", "reason": "CA traffic ratio decreased while DL throughput degraded.", "recommended_action": "Check CA activation, SCell availability, and CA configuration."},
            {"feature": "L.Traffic.ActiveUser.Dl.Avg", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High User Load", "reason": "DL active users increased while DL throughput decreased.", "recommended_action": "Check load, scheduling, congestion, and user distribution."},
        ],
    },

    "UL Throughput": {
        "target_kpi": "(HU) User UL Average Throughput (Mbps)",
        "bad_direction": "low",
        "default_threshold": 20.0,
        "category": "Integrity",
        "output_prefix": "ul_throughput",
        "min_baseline_value": 2.0,  # NEW: Mbps
        "related_rules": [
            {"feature": "(HU)UL PRB Utilization(%)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "UL Congestion", "reason": "UL PRB utilization increased while UL throughput degraded.", "recommended_action": "Check UL load, UL scheduler, and uplink capacity."},
            {"feature": "(HU) Avg UL Interference(dBm)", "bad_direction": "high", "threshold": 10, "severity": 4, "category": "UL Interference Issue", "reason": "UL interference increased while UL throughput degraded.", "recommended_action": "Check external interference, PIM, and uplink noise rise."},
            {"feature": "(HU) UL IBLER(%)", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "High UL Retransmission", "reason": "UL IBLER increased while UL throughput degraded.", "recommended_action": "Check UL BLER, interference, PUSCH MCS, and UE power."},
            {"feature": "UL RBLER", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "UL Radio Block Errors", "reason": "UL RBLER increased while UL throughput degraded.", "recommended_action": "Check UL coverage, interference, and UE power limitation."},
            {"feature": "(HU) PUSCH MCS", "bad_direction": "low", "threshold": 15, "severity": 3, "category": "Low UL Modulation", "reason": "PUSCH MCS decreased while UL throughput degraded.", "recommended_action": "Check uplink SINR, interference, and power control."},
            {"feature": "L.Traffic.ActiveUser.UL.Avg", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High UL User Load", "reason": "UL active users increased while UL throughput decreased.", "recommended_action": "Check uplink scheduling, congestion, and load distribution."},
        ],
    },

    "RRC Setup SR": {
        "target_kpi": "(TE) RRC Setup SR%",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "Accessibility",
        "output_prefix": "rrc_setup_sr",
        "min_baseline_value": 90.0,  # NEW: Only cells with > 90% baseline SR
        "related_rules": [
            {"feature": "L.RRC.ConnReq.Att", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High RRC Attempts", "reason": "RRC attempts increased.", "recommended_action": "Check RACH load, access parameters, admission control, and overload."},
            {"feature": "RRC Setup Failure Time", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "RRC Failure Increase", "reason": "RRC setup failures increased.", "recommended_action": "Check RACH failures, no reply, rejection, admission control, and radio quality."},
            {"feature": "L.RRC.SetupFail.NoReply", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "RRC No Reply", "reason": "RRC no-reply failures increased.", "recommended_action": "Check coverage, interference, RACH configuration, and UE access conditions."},
            {"feature": "L.RRC.SetupFail.Rej", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "RRC Rejection", "reason": "RRC rejection failures increased.", "recommended_action": "Check admission control, overload, forbidden access, and MME overload."},
            {"feature": "L.RRC.SetupFail.Rej.MMEOverload", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "MME Overload", "reason": "RRC rejection due to MME overload increased.", "recommended_action": "Check MME/S1 signaling, core side load, and S1 interface."},
            {"feature": "L.RRC.SetupFail.ResFail", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Radio Resource Failure", "reason": "RRC setup failures due to resource failure increased.", "recommended_action": "Check radio resources, PRB load, admission control, and congestion."},
            {"feature": "RACH Contention-Based Failures", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "RACH Failure", "reason": "RACH contention failures increased.", "recommended_action": "Check PRACH configuration, root sequence planning, preamble load, and coverage."},
        ],
    },

    "ERAB Setup SR": {
        "target_kpi": "ERAB Setup Success Rate",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "Accessibility",
        "output_prefix": "erab_setup_sr",
        "min_baseline_value": 95.0,  # NEW
        "related_rules": [
            {"feature": "L.E-RAB.AttEst", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High ERAB Attempts", "reason": "E-RAB setup attempts increased.", "recommended_action": "Check access load, service attempts, and admission control."},
            {"feature": "ERAB Setup Failure Times", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "ERAB Failure Increase", "reason": "E-RAB setup failures increased.", "recommended_action": "Check ERAB failure reason counters, MME, TNL, RNL, and radio resources."},
            {"feature": "L.E-RAB.FailEst.NoRadioRes", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "No Radio Resource", "reason": "E-RAB failures due to no radio resources increased.", "recommended_action": "Check PRB load, admission control, congestion, and capacity."},
            {"feature": "L.E-RAB.FailEst.NoReply", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "No Reply Failure", "reason": "E-RAB no-reply failures increased.", "recommended_action": "Check radio quality, signaling, and UE response issues."},
            {"feature": "L.E-RAB.FailEst.MME", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "MME Related Failure", "reason": "E-RAB setup failures related to MME increased.", "recommended_action": "Check MME/core side, S1 signaling, and core load."},
            {"feature": "L.E-RAB.FailEst.TNL", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Transport Network Failure", "reason": "E-RAB setup failures related to TNL increased.", "recommended_action": "Check transmission, backhaul, S1-U/S1-C, and transport path."},
        ],
    },

    "Drop Rate": {
        "target_kpi": "E-RAB Drop Rate (E-NodeB + MME) %",
        "bad_direction": "high",
        "default_threshold": 20.0,
        "category": "Retainability",
        "output_prefix": "drop_rate",
        "min_baseline_value": 0.0,  # Any baseline allowed
        "related_rules": [
            {"feature": "L.E-RAB.AbnormRel", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "Abnormal Release Increase", "reason": "E-RAB abnormal releases increased.", "recommended_action": "Check drop reason counters, radio quality, HO failures, and TNL/MME causes."},
            {"feature": "L.E-RAB.AbnormRel.Radio", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "Radio Drop Issue", "reason": "Radio abnormal releases increased.", "recommended_action": "Check coverage, interference, CQI, BLER, and antenna settings."},
            {"feature": "L.E-RAB.AbnormRel.Radio.ULSyncFail", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "UL Sync Failure", "reason": "Drops due to UL synchronization failure increased.", "recommended_action": "Check uplink coverage, UL interference, UE power, and timing advance."},
            {"feature": "L.E-RAB.AbnormRel.Radio.UuNoReply", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Uu No Reply", "reason": "Drops due to Uu no-reply increased.", "recommended_action": "Check coverage holes, interference, and radio link quality."},
            {"feature": "L.E-RAB.AbnormRel.TNL", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Transport Drop Issue", "reason": "Transport-related abnormal releases increased.", "recommended_action": "Check backhaul, transmission alarms, packet loss, and transport congestion."},
            {"feature": "L.E-RAB.AbnormRel.MME", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "MME Drop Issue", "reason": "MME-related abnormal releases increased.", "recommended_action": "Check core network, MME, S1 signaling, and S1 reset counters."},
            {"feature": "L.E-RAB.AbnormRel.HOFailure", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "HO Related Drop", "reason": "Abnormal releases related to HO failure increased.", "recommended_action": "Check neighbors, missing neighbors, A3 offset, CIO, TTT, and PCI issues."},
            {"feature": "RRC Connection Drop Rate%", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "RRC Drop Issue", "reason": "RRC connection drop rate increased.", "recommended_action": "Check radio quality, coverage, interference, re-establishment, and mobility."},
        ],
    },

    "HO Success Rate": {
        "target_kpi": "HO SR% Overall",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "Mobility",
        "output_prefix": "ho_success_rate",
        "min_baseline_value": 90.0,  # NEW
        "related_rules": [
            {"feature": "Intra_Freq HO Prepare Failed Times", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Intra-Frequency HO Preparation Failure", "reason": "Intra-frequency HO preparation failures increased.", "recommended_action": "Check neighbor relations, target cell availability, admission control, and HO prep failure reasons."},
            {"feature": "Intra_Freq HO Execution Failed Times", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "Intra-Frequency HO Execution Failure", "reason": "Intra-frequency HO execution failures increased.", "recommended_action": "Check radio quality, A3 offset, TTT, CIO, PCI, and target cell coverage."},
            {"feature": "Inter_Freq HO Prepare Failed Times", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "Inter-Frequency HO Preparation Failure", "reason": "Inter-frequency HO preparation failures increased.", "recommended_action": "Check inter-frequency neighbors, measurement configuration, frequency priority, and target availability."},
            {"feature": "Inter_Freq HO Execution Failed Times", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "Inter-Frequency HO Execution Failure", "reason": "Inter-frequency HO execution failures increased.", "recommended_action": "Check A3/A5 thresholds, TTT, CIO, target cell coverage, and PCI conflicts."},
            {"feature": "S1 HO Execution Failed Times", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "S1 HO Failure", "reason": "S1 HO execution failures increased.", "recommended_action": "Check S1 handover path, MME, transport, and target eNodeB response."},
            {"feature": "X2 Intra-Freq Failure", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "X2 Intra-Frequency HO Failure", "reason": "X2 intra-frequency HO failures increased.", "recommended_action": "Check X2 links, neighbor relation, target cell, and mobility parameters."},
            {"feature": "X2 Inter-Freq Failure", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "X2 Inter-Frequency HO Failure", "reason": "X2 inter-frequency HO failures increased.", "recommended_action": "Check X2 links, inter-frequency neighbors, and target frequency settings."},
            {"feature": "L.HHO.PingPongHo", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "Ping-Pong HO Issue", "reason": "Ping-pong handovers increased.", "recommended_action": "Tune hysteresis, TTT, CIO, A3 offset, and neighbor priorities."},
        ],
    },

    "Availability": {
        "target_kpi": "Availability",
        "bad_direction": "low",
        "default_threshold": 1.0,
        "category": "Availability",
        "output_prefix": "availability",
        "min_baseline_value": 99.0,  # NEW: Only cells with > 99% baseline availability
        "related_rules": [
            {"feature": "(HU) Cell Unavail Time (s)", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "Cell Unavailable Time Increase", "reason": "Cell unavailable time increased.", "recommended_action": "Check outage, alarms, power, transmission, and site status."},
            {"feature": "L.Cell.Unavail.Dur.Sys(s)", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "System Unavailability", "reason": "System unavailability duration increased.", "recommended_action": "Check system faults, board alarms, transmission, and eNodeB health."},
            {"feature": "L.Cell.Unavail.Dur.Manual(s)", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "Manual Unavailability", "reason": "Manual unavailability duration increased.", "recommended_action": "Check manual lock, planned work, maintenance activity, and cell administrative state."},
            {"feature": "L.Cell.Unavail.Dur.Sys.S1Fail(s)", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "S1 Failure Unavailability", "reason": "S1 failure unavailability duration increased.", "recommended_action": "Check S1 link, MME connection, transmission, and core network alarms."},
        ],
    },

    "RACH Success Rate": {
        "target_kpi": "(HU) RACH Success Rate(%)",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "Accessibility",
        "output_prefix": "rach_success_rate",
        "min_baseline_value": 95.0,  # NEW
        "related_rules": [
            {"feature": "RACH Setup Failed Number", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "RACH Setup Failures", "reason": "RACH setup failures increased.", "recommended_action": "Check PRACH parameters, root sequence planning, coverage, interference, and access load."},
            {"feature": "RACH Contention-Based Failures", "bad_direction": "high", "threshold": 20, "severity": 3, "category": "Contention-Based RACH Failure", "reason": "Contention-based RACH failures increased.", "recommended_action": "Check preamble congestion, PRACH configuration, root sequence, and access load."},
            {"feature": "RACH_att", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High RACH Attempts", "reason": "RACH attempts increased.", "recommended_action": "Check access load, coverage, PRACH capacity, and random access configuration."},
            {"feature": "RACH Contention-Based SR", "bad_direction": "low", "threshold": 5, "severity": 3, "category": "Contention-Based RACH SR Drop", "reason": "Contention-based RACH success rate decreased.", "recommended_action": "Check PRACH configuration, root sequence planning, and access congestion."},
            {"feature": "RACH Non-Contention-Based SR", "bad_direction": "low", "threshold": 5, "severity": 3, "category": "Non-Contention RACH SR Drop", "reason": "Non-contention RACH success rate decreased.", "recommended_action": "Check HO-related RACH, target cell access, and PRACH settings."},
        ],
    },

    "CSFB KPI": {
        "target_kpi": "CSFB SR%",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "CSFB / Voice Accessibility",
        "output_prefix": "csfb_kpi",
        "min_baseline_value": 90.0,  # NEW
        "related_rules": [
            {"feature": "CSFB Failure Times", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "CSFB Failure Increase", "reason": "CSFB failure times increased.", "recommended_action": "Check CSFB failure reasons, MME/S1 signaling, RRC redirection, and target 2G/3G availability."},
            {"feature": "L.CSFB.PrepAtt", "bad_direction": "high", "threshold": 20, "severity": 2, "category": "High CSFB Preparation Attempts", "reason": "CSFB preparation attempts increased, which may increase CSFB load.", "recommended_action": "Check CSFB traffic demand, MME load, S1 signaling, and whether the increase is normal voice demand."},
            {"feature": "L.RRCRedirection.E2W.CSFB", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "E2W CSFB Redirection Drop", "reason": "LTE-to-3G CSFB redirection count decreased compared with baseline.", "recommended_action": "Check UTRAN neighbor configuration, 3G target coverage, UTRAN frequency priority, and CSFB redirection settings."},
            {"feature": "L.RRCRedirection.E2G.CSFB", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "E2G CSFB Redirection Drop", "reason": "LTE-to-2G CSFB redirection count decreased compared with baseline.", "recommended_action": "Check GERAN neighbor configuration, 2G target coverage, GERAN frequency priority, LAI/TAI mapping, and CSFB redirection settings."},
            {"feature": "(TE) RRC Setup SR%", "bad_direction": "low", "threshold": 5, "severity": 4, "category": "LTE RRC Access Issue", "reason": "RRC setup success rate decreased, which can affect CSFB before fallback starts.", "recommended_action": "Check LTE RRC accessibility, RACH, RRC setup failures, admission control, and radio quality."},
            {"feature": "ERAB Setup Success Rate", "bad_direction": "low", "threshold": 5, "severity": 3, "category": "E-RAB Setup Issue", "reason": "E-RAB setup success rate decreased, indicating possible access or core signaling issue affecting services.", "recommended_action": "Check E-RAB setup failures, MME/TNL/RNL causes, admission control, radio resources, and S1 signaling."},
        ],
    },

    "VoLTE KPIs": {
        "target_kpi": "BA_Voice E2E VQI",
        "bad_direction": "low",
        "default_threshold": 5.0,
        "category": "VoLTE",
        "output_prefix": "volte_kpis",
        "min_baseline_value": 3.5,  # NEW: VQI baseline threshold
        "related_rules": [
            {"feature": "VoLTE Traffic (Erl)(Erl)", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "VoLTE Traffic Drop", "reason": "VoLTE traffic decreased.", "recommended_action": "Check VoLTE user demand, IMS service, VoLTE coverage, and QCI-1 traffic."},
            {"feature": "L.Traffic.User.VoIP.Avg", "bad_direction": "low", "threshold": 20, "severity": 2, "category": "VoIP User Drop", "reason": "Average VoIP users decreased.", "recommended_action": "Check VoLTE traffic demand, service availability, and IMS registration behavior."},
            {"feature": "DL Traffic QCI-1", "bad_direction": "low", "threshold": 20, "severity": 3, "category": "QCI-1 DL Traffic Drop", "reason": "QCI-1 DL traffic decreased.", "recommended_action": "Check VoLTE bearer traffic, IMS service, and VoLTE user behavior."},
            {"feature": "E-RAB Drop(ENB+MME)_Tot", "bad_direction": "high", "threshold": 20, "severity": 5, "category": "VoLTE Retainability Risk", "reason": "Total E-RAB drops increased.", "recommended_action": "Check VoLTE drops, radio quality, TNL/MME causes, and mobility."},
            {"feature": "E-RAB Drop Rate QCI 7", "bad_direction": "high", "threshold": 20, "severity": 4, "category": "QCI-7 Drop Issue", "reason": "QCI-7 drop rate increased.", "recommended_action": "Check VoLTE-related bearer retainability and radio quality."},
            {"feature": "BA_Overall SRVCC HO Execution Success Rate (%)", "bad_direction": "low", "threshold": 5, "severity": 4, "category": "SRVCC Execution Degradation", "reason": "SRVCC HO execution success rate decreased.", "recommended_action": "Check SRVCC neighbors, 2G/3G target cells, IMS/SRVCC configuration, and mobility parameters."},
            {"feature": "BA_Overall SRVCC HO Preparation Success Rate (%)", "bad_direction": "low", "threshold": 5, "severity": 3, "category": "SRVCC Preparation Degradation", "reason": "SRVCC HO preparation success rate decreased.", "recommended_action": "Check SRVCC preparation, target availability, MSC/MME coordination, and neighbor definitions."},
        ],
    },
}

print(f"KPI Configuration loaded: {len(KPI_CONFIGS)} KPI categories")

## 4. Helper Functions

In [ ]:
def clean_excel_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Clean Excel column names from spaces and hidden line breaks."""
    df = df.copy()
    cleaned_columns = []
    for col in df.columns:
        col = str(col)
        col = col.replace(chr(10), " ")
        col = col.replace(chr(13), " ")
        col = col.strip()
        cleaned_columns.append(col)
    df.columns = cleaned_columns
    return df


def normalize_column_name(col) -> str:
    """Normalize column names for smart matching."""
    col = str(col).lower()
    col = col.replace(" ", "")
    col = col.replace("_", "")
    col = col.replace("-", "")
    col = col.replace(chr(10), "")
    col = col.replace(chr(13), "")
    col = col.strip()
    return col


def find_matching_column(df: pd.DataFrame, wanted_col: str):
    """Find real Excel column even if spaces/newlines are different."""
    if wanted_col in df.columns:
        return wanted_col
    for col in df.columns:
        if str(col).strip() == str(wanted_col).strip():
            return col
    wanted_norm = normalize_column_name(wanted_col)
    for col in df.columns:
        if normalize_column_name(col) == wanted_norm:
            return col
    return None


def clean_numeric_series(series: pd.Series) -> pd.Series:
    """Convert mixed numeric/text column to numeric - Enhanced with N/A handling."""
    return pd.to_numeric(
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("N/A", "", regex=False)
        .str.replace("#DIV/0!", "", regex=False)
        .str.replace("#N/A", "", regex=False)
        .str.replace("NIL", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce",
    )


def calculate_degradation(recent_value, baseline_value, bad_direction):
    """Return degradation percentage. Positive means worse."""
    if pd.isna(recent_value) or pd.isna(baseline_value):
        return np.nan
    if baseline_value == 0:
        return np.nan
    if bad_direction == "low":
        return ((baseline_value - recent_value) / baseline_value) * 100
    if bad_direction == "high":
        return ((recent_value - baseline_value) / baseline_value) * 100
    return np.nan


def perform_ttest(recent_values, baseline_values):
    """
    Perform Welch's t-test between recent and baseline periods.
    Returns (is_significant, p_value, t_statistic)
    """
    try:
        recent_clean = recent_values.dropna()
        baseline_clean = baseline_values.dropna()
        
        if len(recent_clean) < 2 or len(baseline_clean) < 2:
            return False, np.nan, np.nan
        
        t_stat, p_value = stats.ttest_ind(recent_clean, baseline_clean, equal_var=False)
        
        # Significant if p < 0.05
        is_significant = p_value < 0.05
        return is_significant, p_value, t_stat
    except Exception:
        return False, np.nan, np.nan


def get_periods_enhanced(df, date_col, num_days, baseline_mode=BASELINE_MODE_LAST_WEEK, 
                          custom_baseline_start=None, custom_baseline_end=None):
    """
    Get analysis periods with configurable baseline window.
    
    baseline_mode options:
    - BASELINE_MODE_LAST_WEEK: Same N days from last week (original behavior)
    - BASELINE_MODE_4WEEK_AVG: 4-week rolling average
    - BASELINE_MODE_CUSTOM: User-defined date range
    """
    # Normalize to date level for hourly data
    last_date = df[date_col].dt.normalize().max()
    recent_start = last_date - pd.Timedelta(days=num_days - 1)
    recent_end = last_date
    
    if baseline_mode == BASELINE_MODE_LAST_WEEK:
        baseline_start = recent_start - pd.Timedelta(days=7)
        baseline_end = recent_end - pd.Timedelta(days=7)
        
    elif baseline_mode == BASELINE_MODE_4WEEK_AVG:
        # 4-week rolling: use all 4 weeks before recent period
        baseline_start = recent_start - pd.Timedelta(days=28)
        baseline_end = recent_start - pd.Timedelta(days=1)
        
    elif baseline_mode == BASELINE_MODE_CUSTOM:
        if custom_baseline_start and custom_baseline_end:
            baseline_start = pd.Timestamp(custom_baseline_start)
            baseline_end = pd.Timestamp(custom_baseline_end)
        else:
            # Fallback to last week
            baseline_start = recent_start - pd.Timedelta(days=7)
            baseline_end = recent_end - pd.Timedelta(days=7)
    else:
        baseline_start = recent_start - pd.Timedelta(days=7)
        baseline_end = recent_end - pd.Timedelta(days=7)
    
    return last_date, recent_start, recent_end, baseline_start, baseline_end


print("Helper functions defined!")

## 5. Cause Detection Functions

In [ ]:
def find_degradation_causes_vectorized(df, rules):
    """
    Vectorized cause detection for performance optimization.
    Replaces row-by-row apply() with column-wise operations.
    
    NEW: Uses severity weighting for cause ranking.
    
    IMPORTANT: Reset index before calling this function to ensure proper alignment.
    """
    # CRITICAL FIX: Reset index to ensure proper alignment
    df_work = df.reset_index(drop=True).copy()
    
    detected_causes_list = []
    
    for rule in rules:
        feature = rule["feature"]
        recent_col = f"recent_{feature}"
        baseline_col = f"baseline_{feature}"
        
        if recent_col not in df_work.columns or baseline_col not in df_work.columns:
            continue
        
        recent_values = df_work[recent_col].values  # Use .values for position-based access
        baseline_values = df_work[baseline_col].values
        bad_direction = rule["bad_direction"]
        threshold = rule["threshold"]
        severity = rule.get("severity", 3)  # Default severity
        
        # Vectorized calculation using numpy arrays
        with np.errstate(divide='ignore', invalid='ignore'):
            if bad_direction == "low":
                change_pct = np.where(
                    baseline_values != 0,
                    ((baseline_values - recent_values) / baseline_values) * 100,
                    np.nan
                )
            else:  # high
                change_pct = np.where(
                    baseline_values != 0,
                    ((recent_values - baseline_values) / baseline_values) * 100,
                    np.nan
                )
        
        # Create mask for cells passing threshold
        mask = change_pct >= threshold
        mask = mask & ~np.isnan(change_pct)  # Exclude NaN values
        
        if mask.any():
            # Score = change_pct * severity_weight
            score = change_pct * severity
            
            # Get positions where mask is True
            positions = np.where(mask)[0]
            
            for pos in positions:
                detected_causes_list.append({
                    "row_pos": pos,
                    "feature": feature,
                    "recent_value": recent_values[pos],
                    "baseline_value": baseline_values[pos],
                    "change_pct": change_pct[pos],
                    "severity": severity,
                    "score": score[pos],
                    "category": rule["category"],
                    "reason": rule["reason"],
                    "recommended_action": rule["recommended_action"],
                })
    
    # Default result columns
    default_cols = {
        "main_cause_counter_or_kpi": "No strong related counter detected",
        "main_cause_recent_value": np.nan,
        "main_cause_baseline_value": np.nan,
        "main_cause_change_%": np.nan,
        "main_root_cause_category": "Unknown",
        "main_degradation_reason": "Main KPI degraded, but no related counter passed its threshold.",
        "main_recommended_action": "Check raw counters, alarms, availability, recent changes, and nearby cells manually.",
        "number_of_detected_causes": 0,
        "multi_cause_flag": "No",
        "all_detected_causes": "None",
        "all_cause_categories": "Unknown",
        "all_recommended_actions": "Manual investigation needed",
    }
    
    # If no causes detected, return defaults for all rows
    if not detected_causes_list:
        result_df = pd.DataFrame(default_cols, index=range(len(df_work)))
        return result_df
    
    # Convert to DataFrame
    causes_df = pd.DataFrame(detected_causes_list)
    
    # Sort by score (severity-weighted) for each cell
    causes_df = causes_df.sort_values(["row_pos", "score"], ascending=[True, False])
    
    # Aggregate causes per cell using row position
    result_dict = {}
    
    for row_pos in range(len(df_work)):
        cell_causes = causes_df[causes_df["row_pos"] == row_pos].sort_values("score", ascending=False)
        
        if len(cell_causes) == 0:
            result_dict[row_pos] = default_cols.copy()
        else:
            main_cause = cell_causes.iloc[0]
            
            all_causes_text = " | ".join([
                f"{row['feature']}: recent={row['recent_value']:.2f}, baseline={row['baseline_value']:.2f}, change={row['change_pct']:.2f}%"
                for _, row in cell_causes.head(5).iterrows()
            ])
            all_categories_text = " | ".join(cell_causes["category"].head(5).tolist())
            all_actions_text = " | ".join(cell_causes["recommended_action"].head(5).tolist())
            
            result_dict[row_pos] = {
                "main_cause_counter_or_kpi": main_cause["feature"],
                "main_cause_recent_value": main_cause["recent_value"],
                "main_cause_baseline_value": main_cause["baseline_value"],
                "main_cause_change_%": main_cause["change_pct"],
                "main_root_cause_category": main_cause["category"],
                "main_degradation_reason": main_cause["reason"],
                "main_recommended_action": main_cause["recommended_action"],
                "number_of_detected_causes": len(cell_causes),
                "multi_cause_flag": "Yes" if len(cell_causes) > 1 else "No",
                "all_detected_causes": all_causes_text,
                "all_cause_categories": all_categories_text,
                "all_recommended_actions": all_actions_text,
            }
    
    # Create result DataFrame with proper row positions
    result_df = pd.DataFrame.from_dict(result_dict, orient='index')
    
    return result_df


def find_degradation_causes_row(row, rules):
    """
    Row-by-row cause detection (fallback method).
    Used when vectorized detection fails.
    """
    detected_causes = []
    
    for rule in rules:
        feature = rule["feature"]
        recent_col = f"recent_{feature}"
        baseline_col = f"baseline_{feature}"
        
        if recent_col not in row.index or baseline_col not in row.index:
            continue
        
        recent_value = row[recent_col]
        baseline_value = row[baseline_col]
        
        change_pct = calculate_degradation(
            recent_value,
            baseline_value,
            rule["bad_direction"],
        )
        
        if pd.isna(change_pct):
            continue
        
        if change_pct >= rule["threshold"]:
            severity = rule.get("severity", 3)
            detected_causes.append({
                "feature": feature,
                "recent_value": recent_value,
                "baseline_value": baseline_value,
                "change_pct": change_pct,
                "severity": severity,
                "score": change_pct * severity,
                "category": rule["category"],
                "reason": rule["reason"],
                "recommended_action": rule["recommended_action"],
            })
    
    if not detected_causes:
        return pd.Series({
            "main_cause_counter_or_kpi": "No strong related counter detected",
            "main_cause_recent_value": np.nan,
            "main_cause_baseline_value": np.nan,
            "main_cause_change_%": np.nan,
            "main_root_cause_category": "Unknown",
            "main_degradation_reason": "Main KPI degraded, but no related counter passed its threshold.",
            "main_recommended_action": "Check raw counters, alarms, availability, recent changes, and nearby cells manually.",
            "number_of_detected_causes": 0,
            "multi_cause_flag": "No",
            "all_detected_causes": "None",
            "all_cause_categories": "Unknown",
            "all_recommended_actions": "Manual investigation needed",
        })
    
    # Sort by severity-weighted score
    detected_causes = sorted(detected_causes, key=lambda x: x["score"], reverse=True)
    main_cause = detected_causes[0]
    
    all_causes_text = " | ".join([
        f"{c['feature']}: recent={c['recent_value']:.2f}, baseline={c['baseline_value']:.2f}, change={c['change_pct']:.2f}%"
        for c in detected_causes[:5]
    ])
    all_categories_text = " | ".join([c["category"] for c in detected_causes[:5]])
    all_actions_text = " | ".join([c["recommended_action"] for c in detected_causes[:5]])
    
    return pd.Series({
        "main_cause_counter_or_kpi": main_cause["feature"],
        "main_cause_recent_value": main_cause["recent_value"],
        "main_cause_baseline_value": main_cause["baseline_value"],
        "main_cause_change_%": main_cause["change_pct"],
        "main_root_cause_category": main_cause["category"],
        "main_degradation_reason": main_cause["reason"],
        "main_recommended_action": main_cause["recommended_action"],
        "number_of_detected_causes": len(detected_causes),
        "multi_cause_flag": "Yes" if len(detected_causes) > 1 else "No",
        "all_detected_causes": all_causes_text,
        "all_cause_categories": all_categories_text,
        "all_recommended_actions": all_actions_text,
    })


print("Cause detection functions defined!")

## 6. Main Analysis Engine

In [ ]:
def analyze_selected_kpi_enhanced(
    df, 
    selected_kpi_name, 
    num_days, 
    degradation_threshold, 
    require_complete_days=True,
    baseline_mode=BASELINE_MODE_LAST_WEEK,
    custom_baseline_start=None,
    custom_baseline_end=None,
    enable_significance_test=True,
    log_callback=None
):
    """
    Enhanced analysis with:
    - Configurable baseline window
    - Minimum baseline value filter
    - Statistical significance test
    - Max/percentile aggregation for failure counters
    - Zero baseline warning
    
    Returns: (output_df, metadata)
    """
    def log_msg(msg):
        if log_callback:
            log_callback(msg)
        else:
            print(msg)
    
    config = KPI_CONFIGS[selected_kpi_name]
    
    # Clean Excel column names
    df = clean_excel_columns(df)
    
    # Smart target KPI matching
    original_target_kpi = config["target_kpi"]
    target_kpi = find_matching_column(df, original_target_kpi)
    
    if target_kpi is None:
        raise ValueError(f"Target KPI column not found in Excel: {original_target_kpi}")
    
    bad_direction = config["bad_direction"]
    related_rules = config["related_rules"]
    min_baseline_value = config.get("min_baseline_value", 0.0)
    
    needed_cols = CELL_ID_COLS + [DATE_COL, target_kpi]
    missing_cols = [col for col in needed_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    df_kpi = df[needed_cols].copy()
    # Normalize date to handle hourly data
    df_kpi[DATE_COL] = pd.to_datetime(df_kpi[DATE_COL], errors="coerce").dt.normalize()
    df_kpi[target_kpi] = clean_numeric_series(df_kpi[target_kpi])
    df_kpi = df_kpi.dropna(subset=[DATE_COL, target_kpi])
    df_kpi = df_kpi[df_kpi[target_kpi] >= 0].copy()
    
    # Get periods with enhanced baseline mode
    last_date, recent_start, recent_end, baseline_start, baseline_end = get_periods_enhanced(
        df_kpi, DATE_COL, num_days, baseline_mode, custom_baseline_start, custom_baseline_end
    )
    
    recent_df = df_kpi[(df_kpi[DATE_COL] >= recent_start) & (df_kpi[DATE_COL] <= recent_end)].copy()
    baseline_df = df_kpi[(df_kpi[DATE_COL] >= baseline_start) & (df_kpi[DATE_COL] <= baseline_end)].copy()
    
    # Enhanced aggregation with max and percentile for failure counters
    agg_funcs = {
        target_kpi: ["mean", "max", "sum"]
    }
    
    recent_agg = recent_df.groupby(CELL_ID_COLS).agg({
        target_kpi: ["mean", "max", "sum"],
        DATE_COL: "nunique"
    }).reset_index()
    recent_agg.columns = CELL_ID_COLS + ["recent_avg_kpi", "recent_max_kpi", "recent_total_kpi", "recent_days_count"]
    
    baseline_agg = baseline_df.groupby(CELL_ID_COLS).agg({
        target_kpi: ["mean", "max", "sum"],
        DATE_COL: "nunique"
    }).reset_index()
    baseline_agg.columns = CELL_ID_COLS + ["baseline_avg_kpi", "baseline_max_kpi", "baseline_total_kpi", "baseline_days_count"]
    
    comparison = recent_agg.merge(baseline_agg, on=CELL_ID_COLS, how="inner")
    
    if require_complete_days:
        comparison = comparison[
            (comparison["recent_days_count"] == num_days) &
            (comparison["baseline_days_count"] == num_days)
        ].copy()
    
    # NEW: Log warning for zero baseline cells
    zero_baseline_mask = comparison["baseline_avg_kpi"] == 0
    zero_baseline_count = zero_baseline_mask.sum()
    if zero_baseline_count > 0:
        zero_baseline_cells = comparison[zero_baseline_mask][CELL_COL].head(10).tolist()
        log_msg(f"WARNING: {zero_baseline_count} cells have zero baseline value and will be excluded")
        log_msg(f"Examples: {zero_baseline_cells}")
    
    comparison = comparison[comparison["baseline_avg_kpi"] != 0].copy()
    
    # NEW: Apply minimum baseline value filter
    excluded_by_min = 0
    if min_baseline_value > 0:
        min_baseline_mask = comparison["baseline_avg_kpi"] >= min_baseline_value
        excluded_by_min = (~min_baseline_mask).sum()
        if excluded_by_min > 0:
            log_msg(f"INFO: {excluded_by_min} cells excluded by min_baseline_value filter (< {min_baseline_value})")
        comparison = comparison[min_baseline_mask].copy()
    
    # Calculate degradation
    comparison["kpi_degradation_ratio_%"] = comparison.apply(
        lambda row: calculate_degradation(row["recent_avg_kpi"], row["baseline_avg_kpi"], bad_direction),
        axis=1,
    )
    
    # NEW: Statistical significance test
    if enable_significance_test:
        significance_results = []
        for idx, row in comparison.iterrows():
            cell_id = (row[SITE_COL], row[CELL_COL])
            
            # Get daily values for this cell
            cell_recent = recent_df[
                (recent_df[SITE_COL] == cell_id[0]) & 
                (recent_df[CELL_COL] == cell_id[1])
            ][target_kpi]
            cell_baseline = baseline_df[
                (baseline_df[SITE_COL] == cell_id[0]) & 
                (baseline_df[CELL_COL] == cell_id[1])
            ][target_kpi]
            
            is_sig, p_val, t_stat = perform_ttest(cell_recent, cell_baseline)
            significance_results.append({
                "index": idx,
                "stat_significant": is_sig,
                "p_value": p_val,
                "t_statistic": t_stat
            })
        
        sig_df = pd.DataFrame(significance_results).set_index("index")
        comparison["stat_significant"] = sig_df["stat_significant"].reindex(comparison.index).fillna(False)
        comparison["p_value"] = sig_df["p_value"].reindex(comparison.index)
        comparison["t_statistic"] = sig_df["t_statistic"].reindex(comparison.index)
    
    comparison["kpi_status"] = np.where(
        comparison["kpi_degradation_ratio_%"] >= degradation_threshold,
        "Degraded",
        "Normal",
    )
    
    # If significance test enabled, require both threshold AND significance
    if enable_significance_test:
        comparison["kpi_status"] = np.where(
            (comparison["kpi_degradation_ratio_%"] >= degradation_threshold) & 
            (comparison["stat_significant"] == True),
            "Degraded",
            "Normal",
        )
    
    comparison["selected_kpi_name"] = selected_kpi_name
    comparison["target_kpi_column"] = target_kpi
    comparison["kpi_category"] = config["category"]
    comparison["kpi_bad_direction"] = bad_direction
    comparison["selected_threshold_%"] = degradation_threshold
    comparison["recent_period"] = f"{recent_start.date()} to {recent_end.date()}"
    comparison["baseline_period"] = f"{baseline_start.date()} to {baseline_end.date()}"
    comparison["baseline_mode"] = baseline_mode
    
    degraded_cells = comparison[comparison["kpi_status"] == "Degraded"].copy()
    degraded_cells = degraded_cells.sort_values("kpi_degradation_ratio_%", ascending=False)
    
    debug_info = {
        "cells_after_merge": comparison.shape[0],
        "recent_days_distribution": comparison["recent_days_count"].value_counts().sort_index().to_dict(),
        "baseline_days_distribution": comparison["baseline_days_count"].value_counts().sort_index().to_dict(),
        "max_degradation": comparison["kpi_degradation_ratio_%"].max() if not comparison.empty else None,
        "min_degradation": comparison["kpi_degradation_ratio_%"].min() if not comparison.empty else None,
        "mean_degradation": comparison["kpi_degradation_ratio_%"].mean() if not comparison.empty else None,
        "zero_baseline_excluded": zero_baseline_count,
        "min_baseline_excluded": excluded_by_min if min_baseline_value > 0 else 0,
    }
    
    metadata = {
        "last_date": last_date,
        "recent_start": recent_start,
        "recent_end": recent_end,
        "baseline_start": baseline_start,
        "baseline_end": baseline_end,
        "baseline_mode": baseline_mode,
        "available_related_features": [],
        "missing_related_features": [],
        "debug_info": debug_info,
    }
    
    if degraded_cells.empty:
        return degraded_cells, metadata
    
    # Smart related counter matching
    available_related_rules = []
    missing_related_features = []
    
    for rule in related_rules:
        matched_col = find_matching_column(df, rule["feature"])
        if matched_col is not None:
            new_rule = rule.copy()
            new_rule["feature"] = matched_col
            available_related_rules.append(new_rule)
        else:
            missing_related_features.append(rule["feature"])
    
    available_related_features = [rule["feature"] for rule in available_related_rules]
    metadata["available_related_features"] = available_related_features
    metadata["missing_related_features"] = missing_related_features
    
    if available_related_features:
        reason_cols = CELL_ID_COLS + [DATE_COL] + available_related_features
        df_reason = df[reason_cols].copy()
        df_reason[DATE_COL] = pd.to_datetime(df_reason[DATE_COL], errors="coerce").dt.normalize()
        
        for col in available_related_features:
            df_reason[col] = clean_numeric_series(df_reason[col])
        
        recent_reason_df = df_reason[(df_reason[DATE_COL] >= recent_start) & (df_reason[DATE_COL] <= recent_end)].copy()
        baseline_reason_df = df_reason[(df_reason[DATE_COL] >= baseline_start) & (df_reason[DATE_COL] <= baseline_end)].copy()
        
        # Enhanced aggregation: mean + max for failure counters
        agg_dict = {}
        for col in available_related_features:
            agg_dict[col] = ["mean", "max"]
        
        recent_reason_agg = recent_reason_df.groupby(CELL_ID_COLS).agg(agg_dict).reset_index()
        baseline_reason_agg = baseline_reason_df.groupby(CELL_ID_COLS).agg(agg_dict).reset_index()
        
        # Flatten column names
        new_cols = CELL_ID_COLS.copy()
        for col in available_related_features:
            new_cols.append(f"recent_{col}_mean")
            new_cols.append(f"recent_{col}_max")
        recent_reason_agg.columns = new_cols
        
        new_cols = CELL_ID_COLS.copy()
        for col in available_related_features:
            new_cols.append(f"baseline_{col}_mean")
            new_cols.append(f"baseline_{col}_max")
        baseline_reason_agg.columns = new_cols
        
        # Create aliases for compatibility with cause detection
        for col in available_related_features:
            recent_reason_agg[f"recent_{col}"] = recent_reason_agg[f"recent_{col}_mean"]
            baseline_reason_agg[f"baseline_{col}"] = baseline_reason_agg[f"baseline_{col}_mean"]
        
        degraded_with_causes = degraded_cells.merge(recent_reason_agg, on=CELL_ID_COLS, how="left")
        degraded_with_causes = degraded_with_causes.merge(baseline_reason_agg, on=CELL_ID_COLS, how="left")
        
        # CRITICAL: Reset index for proper alignment with cause detection
        degraded_with_causes = degraded_with_causes.reset_index(drop=True)
        
        # Use vectorized cause detection with fallback
        try:
            cause_results = find_degradation_causes_vectorized(degraded_with_causes, available_related_rules)
            degraded_with_causes = pd.concat([degraded_with_causes.reset_index(drop=True), cause_results.reset_index(drop=True)], axis=1)
        except Exception as vec_error:
            # Fallback to row-by-row apply if vectorized fails
            log_msg(f"Vectorized cause detection failed, using fallback: {vec_error}")
            cause_results = degraded_with_causes.apply(
                lambda row: find_degradation_causes_row(row, available_related_rules),
                axis=1,
            )
            degraded_with_causes = pd.concat([degraded_with_causes, cause_results], axis=1)
    else:
        degraded_with_causes = degraded_cells.copy()
        degraded_with_causes["main_cause_counter_or_kpi"] = "No related counters available in sheet"
        degraded_with_causes["main_cause_recent_value"] = np.nan
        degraded_with_causes["main_cause_baseline_value"] = np.nan
        degraded_with_causes["main_cause_change_%"] = np.nan
        degraded_with_causes["main_root_cause_category"] = "Unknown"
        degraded_with_causes["main_degradation_reason"] = "No related counters from the config were found in the uploaded sheet."
        degraded_with_causes["main_recommended_action"] = "Check KPI manually or update KPI_CONFIGS with available counters."
        degraded_with_causes["number_of_detected_causes"] = 0
        degraded_with_causes["multi_cause_flag"] = "No"
        degraded_with_causes["all_detected_causes"] = "None"
        degraded_with_causes["all_cause_categories"] = "Unknown"
        degraded_with_causes["all_recommended_actions"] = "Manual investigation needed"
    
    final_cols = CELL_ID_COLS + [
        "selected_kpi_name", "target_kpi_column", "kpi_category", "kpi_bad_direction",
        "selected_threshold_%", "recent_period", "baseline_period", "baseline_mode",
        "recent_avg_kpi", "baseline_avg_kpi", "recent_max_kpi", "baseline_max_kpi",
        "recent_total_kpi", "baseline_total_kpi", "recent_days_count", "baseline_days_count",
        "kpi_degradation_ratio_%", "kpi_status", "stat_significant", "p_value",
        "main_cause_counter_or_kpi", "main_cause_recent_value", "main_cause_baseline_value",
        "main_cause_change_%", "main_root_cause_category", "main_degradation_reason",
        "main_recommended_action", "number_of_detected_causes", "multi_cause_flag",
        "all_detected_causes", "all_cause_categories", "all_recommended_actions",
    ]
    
    # Filter to available columns
    available_final_cols = [col for col in final_cols if col in degraded_with_causes.columns]
    
    return degraded_with_causes[available_final_cols].copy(), metadata


print("Main analysis engine defined!")

## 7. Usage Example: Load and Analyze Data

In [ ]:
# Example: Load your Excel file
# Replace the file path with your actual data file

file_path = "path/to/your/kpi_data.xlsx"  # Change this to your file path
sheet_name = 0  # or specify sheet name like "Sheet1"

# Load the data
# df = pd.read_excel(file_path, sheet_name=sheet_name)
# print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
# df.head()

## 8. Run Single KPI Analysis

In [ ]:
# Configure analysis parameters
selected_kpi = "DL Traffic"  # Choose from KPI_CONFIGS.keys()
num_days = 4  # Number of days for comparison
threshold = 30.0  # Degradation threshold percentage
baseline_mode = BASELINE_MODE_LAST_WEEK  # Options: BASELINE_MODE_LAST_WEEK, BASELINE_MODE_4WEEK_AVG, BASELINE_MODE_CUSTOM
enable_significance_test = True  # Enable t-test significance filter

# Run analysis (uncomment when df is loaded)
# output_df, metadata = analyze_selected_kpi_enhanced(
#     df=df,
#     selected_kpi_name=selected_kpi,
#     num_days=num_days,
#     degradation_threshold=threshold,
#     require_complete_days=True,
#     baseline_mode=baseline_mode,
#     enable_significance_test=enable_significance_test,
#     log_callback=print
# )

# print(f"\n=== Analysis Results ===")
# print(f"Recent Period: {metadata['recent_start'].date()} to {metadata['recent_end'].date()}")
# print(f"Baseline Period: {metadata['baseline_start'].date()} to {metadata['baseline_end'].date()}")
# print(f"Degraded Cells Found: {len(output_df)}")
# print(f"\nMissing Features: {metadata['missing_related_features']}")

# # Display results
# if not output_df.empty:
#     display(output_df.head(10))

## 9. Analyze All KPIs

In [ ]:
def analyze_all_kpis(df, num_days=4, baseline_mode=BASELINE_MODE_LAST_WEEK, 
                     enable_significance_test=True, log_callback=print):
    """
    Analyze all KPIs and return combined results.
    """
    outputs = {}
    summary_records = []
    all_kpi_names = list(KPI_CONFIGS.keys())
    
    for idx, kpi_name in enumerate(all_kpi_names, 1):
        config = KPI_CONFIGS[kpi_name]
        threshold = float(config["default_threshold"])
        
        log_callback(f"Analyzing {idx}/{len(all_kpi_names)}: {kpi_name}")
        
        try:
            output_df, metadata = analyze_selected_kpi_enhanced(
                df=df,
                selected_kpi_name=kpi_name,
                num_days=num_days,
                degradation_threshold=threshold,
                require_complete_days=True,
                baseline_mode=baseline_mode,
                enable_significance_test=enable_significance_test,
                log_callback=log_callback,
            )
            outputs[kpi_name] = output_df
            
            debug = metadata.get("debug_info", {})
            degraded_count = output_df.shape[0]
            
            summary_records.append({
                "kpi_name": kpi_name,
                "target_kpi_column": config["target_kpi"],
                "kpi_category": config["category"],
                "threshold_%": threshold,
                "degraded_cells_count": degraded_count,
                "max_degradation_%": debug.get("max_degradation"),
                "mean_degradation_%": debug.get("mean_degradation"),
                "status": "Completed",
                "error": ""
            })
            
        except Exception as e:
            outputs[kpi_name] = pd.DataFrame()
            summary_records.append({
                "kpi_name": kpi_name,
                "target_kpi_column": config.get("target_kpi", ""),
                "kpi_category": config.get("category", ""),
                "threshold_%": threshold,
                "degraded_cells_count": 0,
                "max_degradation_%": None,
                "mean_degradation_%": None,
                "status": "Failed",
                "error": str(e)
            })
            log_callback(f"  ERROR: {e}")
    
    # Combine all results
    non_empty = [df for df in outputs.values() if df is not None and not df.empty]
    combined = pd.concat(non_empty, ignore_index=True) if non_empty else pd.DataFrame()
    summary_df = pd.DataFrame(summary_records)
    
    return combined, summary_df, outputs


# Run all KPI analysis (uncomment when df is loaded)
# combined_df, summary_df, all_outputs = analyze_all_kpis(
#     df, 
#     num_days=4, 
#     baseline_mode=BASELINE_MODE_LAST_WEEK,
#     enable_significance_test=True
# )

# print("\n=== Summary ===")
# display(summary_df)

## 10. Visualization Functions

In [ ]:
def plot_degraded_cells_per_kpi(summary_df, figsize=(12, 6)):
    """Plot bar chart of degraded cells per KPI."""
    if summary_df is None or summary_df.empty:
        print("No summary data available")
        return
    
    plot_df = summary_df.sort_values("degraded_cells_count", ascending=False)
    
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(plot_df["kpi_name"], plot_df["degraded_cells_count"], color='steelblue')
    
    ax.set_xlabel('KPI Category')
    ax.set_ylabel('Number of Degraded Cells')
    ax.set_title('Degraded Cells per KPI Category')
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{int(height)}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()


def plot_root_cause_distribution(output_df, figsize=(10, 6)):
    """Plot horizontal bar chart of root cause distribution."""
    if output_df is None or output_df.empty:
        print("No output data available")
        return
    
    if "main_root_cause_category" not in output_df.columns:
        print("Root cause category column not found")
        return
    
    causes = output_df["main_root_cause_category"].value_counts().head(15).sort_values()
    
    fig, ax = plt.subplots(figsize=figsize)
    causes.plot(kind='barh', ax=ax, color='coral')
    
    ax.set_xlabel('Number of Cells')
    ax.set_ylabel('Root Cause Category')
    ax.set_title('Root Cause Distribution')
    
    plt.tight_layout()
    plt.show()


def plot_kpi_trend(df, kpi_column, date_col=DATE_COL, cell_id_cols=CELL_ID_COLS,
                   site_col=SITE_COL, cell_col=CELL_COL, degraded_cells=None, figsize=(14, 5)):
    """
    Plot KPI trend before and after removing degraded cells.
    """
    if kpi_column not in df.columns:
        print(f"KPI column '{kpi_column}' not found in data")
        return
    
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    
    # Daily average before removal (all cells)
    daily_before = df.groupby(date_col)[kpi_column].mean().reset_index()
    daily_before.columns = ['Date', 'Average']
    
    # Daily average after removal (clean cells)
    if degraded_cells and len(degraded_cells) > 0:
        mask = df.set_index([site_col, cell_col]).index.isin(degraded_cells)
        df_clean = df[~mask]
        daily_after = df_clean.groupby(date_col)[kpi_column].mean().reset_index() if len(df_clean) > 0 else daily_before.copy()
        daily_after.columns = ['Date', 'Average']
    else:
        daily_after = daily_before.copy()
    
    if daily_before.empty:
        print("No data to plot")
        return
    
    fig, ax = plt.subplots(figsize=figsize)
    
    dates = daily_before['Date'].tolist()
    x = range(len(dates))
    labels = [d.strftime('%m/%d') if hasattr(d, 'strftime') else str(d)[:10] for d in dates]
    
    ax.plot(x, daily_before['Average'].values, 'b-o', linewidth=2, markersize=6, label='Before (All Cells)')
    ax.plot(x, daily_after['Average'].values, 'g-s', linewidth=2, markersize=6, label='After (Clean Cells)')
    
    if degraded_cells and len(degraded_cells) > 0:
        diff = daily_before['Average'].values - daily_after['Average'].values
        ax.fill_between(x, daily_before['Average'].values, daily_after['Average'].values,
                        where=diff != 0, alpha=0.3, color='red', label='Degraded Impact')
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_xlabel('Date')
    ax.set_ylabel(kpi_column[:40])
    ax.set_title(f'{kpi_column[:40]} - Daily Trend')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


print("Visualization functions defined!")

## 11. Generate Visualizations (After Running Analysis)

In [ ]:
# Plot degraded cells per KPI (uncomment after running analyze_all_kpis)
# plot_degraded_cells_per_kpi(summary_df)

In [ ]:
# Plot root cause distribution (uncomment after running analysis)
# plot_root_cause_distribution(combined_df)

In [ ]:
# Plot KPI trend (uncomment after loading df and running analysis)
# config = KPI_CONFIGS.get("DL Traffic", {})
# target_kpi = config.get("target_kpi", "")
# kpi_col = find_matching_column(df, target_kpi)

# # Get degraded cell IDs
# degraded_cell_ids = set()
# if not combined_df.empty and SITE_COL in combined_df.columns and CELL_COL in combined_df.columns:
#     for _, row in combined_df.iterrows():
#         degraded_cell_ids.add((row.get(SITE_COL, ''), row.get(CELL_COL, '')))

# plot_kpi_trend(df, kpi_col, degraded_cells=degraded_cell_ids)

## 12. Save Results to CSV

In [ ]:
def save_results(output_df, summary_df, all_outputs, output_dir="./output"):
    """Save analysis results to CSV files."""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    saved_files = []
    
    # Save individual KPI results
    for kpi_name, kpi_df in all_outputs.items():
        if kpi_df is not None and not kpi_df.empty:
            prefix = KPI_CONFIGS[kpi_name]["output_prefix"]
            path = os.path.join(output_dir, f"{prefix}_degraded.csv")
            kpi_df.to_csv(path, index=False, encoding="utf-8-sig")
            saved_files.append(path)
    
    # Save combined results
    if output_df is not None and not output_df.empty:
        path = os.path.join(output_dir, "all_kpis_combined.csv")
        output_df.to_csv(path, index=False, encoding="utf-8-sig")
        saved_files.append(path)
    
    # Save summary
    if summary_df is not None and not summary_df.empty:
        path = os.path.join(output_dir, "summary_report.csv")
        summary_df.to_csv(path, index=False, encoding="utf-8-sig")
        saved_files.append(path)
    
    print(f"Saved {len(saved_files)} files:")
    for f in saved_files:
        print(f"  - {f}")
    
    return saved_files


# Save results (uncomment after running analysis)
# save_results(combined_df, summary_df, all_outputs, output_dir="./output")

## 13. Generate Word Report

In [ ]:
def generate_word_report(output_df, summary_df, selected_kpi=None, analysis_mode="all",
                         baseline_mode=BASELINE_MODE_LAST_WEEK, enable_significance_test=True,
                         save_path="RF_Optimization_Report.docx"):
    """
    Generate a Word document report with analysis results.
    """
    if not DOCX_AVAILABLE:
        print("Error: python-docx not installed. Install with: pip install python-docx")
        return None
    
    if output_df is None or (output_df.empty and (summary_df is None or summary_df.empty)):
        print("Error: No results to report. Run analysis first.")
        return None
    
    try:
        doc = Document()
        
        # Title
        doc.add_heading('RF Optimization Analysis Report', 0)
        doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        doc.add_paragraph("Developed by: Musketeers_Team (ITI Graduation Project 2026)")
        doc.add_paragraph(f"Version: 2.0 Enhanced")
        
        # Analysis Summary
        doc.add_heading('Analysis Summary', level=1)
        
        if analysis_mode == "all":
            doc.add_paragraph(f"Analysis Mode: All KPIs Analysis")
            doc.add_paragraph(f"Baseline Mode: {baseline_mode}")
            doc.add_paragraph(f"Significance Test: {'Enabled' if enable_significance_test else 'Disabled'}")
            if summary_df is not None:
                doc.add_paragraph(f"Total KPIs Analyzed: {len(summary_df)}")
                doc.add_paragraph(f"Total Degraded Cells: {summary_df['degraded_cells_count'].sum()}")
        else:
            doc.add_paragraph(f"Analysis Mode: Single KPI Analysis")
            doc.add_paragraph(f"Selected KPI: {selected_kpi}")
            doc.add_paragraph(f"Baseline Mode: {baseline_mode}")
            doc.add_paragraph(f"Degraded Cells: {len(output_df) if output_df is not None else 0}")
        
        # Degraded Cells Table
        if output_df is not None and not output_df.empty:
            doc.add_heading('Degraded Cells Details', level=1)
            
            key_cols = ['eNodeB Name', 'Cell Name', 'kpi_degradation_ratio_%', 
                       'main_root_cause_category', 'main_recommended_action', 'stat_significant', 'p_value']
            available = [c for c in key_cols if c in output_df.columns]
            
            if available:
                table = doc.add_table(rows=1, cols=len(available))
                table.style = 'Table Grid'
                
                headers = table.rows[0].cells
                for i, col in enumerate(available):
                    headers[i].text = col.replace('_', ' ').title()
                
                for _, row in output_df.head(30).iterrows():
                    cells = table.add_row().cells
                    for i, col in enumerate(available):
                        val = row.get(col, '')
                        cells[i].text = "N/A" if pd.isna(val) else str(val)[:80]
        
        # Summary Table
        if analysis_mode == "all" and summary_df is not None and not summary_df.empty:
            doc.add_heading('KPI Summary Table', level=1)
            
            sum_cols = ['kpi_name', 'degraded_cells_count', 'max_degradation_%', 'status']
            avail_sum = [c for c in sum_cols if c in summary_df.columns]
            
            if avail_sum:
                table = doc.add_table(rows=1, cols=len(avail_sum))
                table.style = 'Table Grid'
                
                headers = table.rows[0].cells
                for i, col in enumerate(avail_sum):
                    headers[i].text = col.replace('_', ' ').title()
                
                for _, row in summary_df.iterrows():
                    cells = table.add_row().cells
                    for i, col in enumerate(avail_sum):
                        val = row.get(col, '')
                        cells[i].text = "N/A" if pd.isna(val) else str(val)[:50]
        
        doc.save(save_path)
        print(f"Report saved: {save_path}")
        return save_path
        
    except Exception as e:
        print(f"Error generating report: {e}")
        return None


# Generate report (uncomment after running analysis)
# generate_word_report(
#     output_df=combined_df, 
#     summary_df=summary_df, 
#     analysis_mode="all",
#     baseline_mode=baseline_mode,
#     enable_significance_test=enable_significance_test,
#     save_path="RF_Optimization_Report.docx"
# )

---

## Complete Workflow Example

Below is a complete example showing how to use this notebook:

```python
# 1. Load your data
df = pd.read_excel("your_kpi_data.xlsx", sheet_name=0)

# 2. Run all KPIs analysis
combined_df, summary_df, all_outputs = analyze_all_kpis(
    df, 
    num_days=4, 
    baseline_mode=BASELINE_MODE_LAST_WEEK,
    enable_significance_test=True
)

# 3. Visualize results
plot_degraded_cells_per_kpi(summary_df)
plot_root_cause_distribution(combined_df)

# 4. Save results
save_results(combined_df, summary_df, all_outputs, output_dir="./output")

# 5. Generate Word report
generate_word_report(combined_df, summary_df, save_path="RF_Optimization_Report.docx")
```

---

**Developed by: Musketeers_Team (ITI Graduation Project 2026)**